In [1]:
# !pip install torch torchaudio jiwer pyctcdecode pyspellchecker --quiet
# !pip install https://github.com/kpu/kenlm/archive/master.zip --quiet
# # Force upgrade back to 2.x to satisfy PyTorch's C-backend
# !pip install "numpy>=2.0.0" --upgrade --quiet

In [2]:
!pip install jiwer pyctcdecode pyspellchecker jellyfish tqdm --quiet
!pip install kenlm --quiet
!pip install "numpy>=2.0.0" --upgrade --quiet
print('✅ Python dependencies installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 51.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2

In [3]:
import os
TEST_CLEAN_PATH = '/kaggle/input/datasets/yasiashpot/librispeech/LibriSpeech/test-clean'  # if already a folder
WORK_DIR         = '/kaggle/working'
CHECKPOINT_PATH = '/kaggle/input/models/laughtime/speech2text/pytorch/default/1/best_model.pt'
print(TEST_CLEAN_PATH, os.path.exists(TEST_CLEAN_PATH))
print(CHECKPOINT_PATH, os.path.exists(CHECKPOINT_PATH))
LM_CORPUS_LINES  = 3_000_000

/kaggle/input/datasets/yasiashpot/librispeech/LibriSpeech/test-clean True
/kaggle/input/models/laughtime/speech2text/pytorch/default/1/best_model.pt True


In [4]:
import os

def collect_dataset_info(base_path):
    records = []
    for speaker_id in os.listdir(base_path):
        speaker_path = os.path.join(base_path, speaker_id)
        if not os.path.isdir(speaker_path):
            continue
        for chapter_id in os.listdir(speaker_path):
            chapter_path = os.path.join(speaker_path, chapter_id)
            trans_file = os.path.join(chapter_path, f'{speaker_id}-{chapter_id}.trans.txt')
            if not os.path.exists(trans_file):
                continue
            with open(trans_file) as f:
                for line in f:
                    parts = line.strip().split(' ', 1)
                    if len(parts) != 2:
                        continue
                    utt_id, transcript = parts
                    audio_file = os.path.join(chapter_path, f'{utt_id}.flac')
                    if os.path.exists(audio_file):
                        records.append({'audio_path': audio_file, 'transcript': transcript})
    return records

val_records = collect_dataset_info(TEST_CLEAN_PATH)
print(f'Loaded {len(val_records)} test-clean utterances')

Loaded 2620 test-clean utterances


In [5]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio, torchaudio.transforms as T

SAMPLE_RATE, N_MELS, N_FFT, HOP_LENGTH, WIN_LENGTH = 16000, 128, 512, 160, 400

class TextTransform:
    def __init__(self):
        chars = ['<blank>'] + list('abcdefghijklmnopqrstuvwxyz') + [' ', "'"]
        self.char2idx  = {c: i for i, c in enumerate(chars)}
        self.idx2char  = {i: c for c, i in self.char2idx.items()}
        self.blank_idx = 0
    def int_to_text(self, indices):
        return ''.join(self.idx2char.get(i, '') for i in indices if i != self.blank_idx)
    def __len__(self): return len(self.char2idx)

class LogMelSpectrogram(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=N_MELS,
                                    n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    def forward(self, x):
        x = self.mel(x); x = torch.log(x + 1e-9)
        return (x - x.mean()) / (x.std() + 1e-9)

class CNNLayerNorm(nn.Module):
    def __init__(self, n_feats):
        super().__init__(); self.layer_norm = nn.LayerNorm(n_feats)
    def forward(self, x):
        x = x.transpose(2, 3).contiguous(); x = self.layer_norm(x)
        return x.transpose(2, 3).contiguous()

class ResidualCNN(nn.Module):
    def __init__(self, in_c, out_c, kernel, stride, n_feats, dropout):
        super().__init__()
        self.layer_norm1 = CNNLayerNorm(n_feats)
        self.cnn1 = nn.Conv2d(in_c, out_c, kernel, stride=stride, padding=kernel // 2)
        self.dropout1 = nn.Dropout(dropout)
        self.layer_norm2 = CNNLayerNorm(n_feats)
        self.cnn2 = nn.Conv2d(out_c, out_c, kernel, stride=stride, padding=kernel // 2)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x):
        residual = x
        x = self.layer_norm1(x); x = F.gelu(x); x = self.dropout1(x); x = self.cnn1(x)
        x = self.layer_norm2(x); x = F.gelu(x); x = self.dropout2(x); x = self.cnn2(x)
        return x + residual

class BidirectionalGRU(nn.Module):
    def __init__(self, rnn_dim, hidden_size, dropout, batch_first=False):
        super().__init__()
        self.BiGRU = nn.GRU(rnn_dim, hidden_size, num_layers=1, batch_first=batch_first, bidirectional=True)
        self.layer_norm = nn.LayerNorm(rnn_dim); self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = self.layer_norm(x); x = F.gelu(x); x, _ = self.BiGRU(x); return self.dropout(x)

class SpeechRecognitionModel(nn.Module):
    def __init__(self, n_cnn_layers, n_rnn_layers, rnn_dim, n_class, n_feats, stride=2, dropout=0.1):
        super().__init__()
        self.cnn = nn.Conv2d(1, 32, kernel_size=3, stride=stride, padding=1)
        self.rescnn_layers = nn.Sequential(*[
            ResidualCNN(32, 32, 3, 1, n_feats, dropout + i * 0.05) for i in range(n_cnn_layers)
        ])
        self.fully_connected = nn.Linear(32 * n_feats, rnn_dim)
        self.birnn_layers = nn.Sequential(
            BidirectionalGRU(rnn_dim, rnn_dim, dropout, batch_first=True),
            *[BidirectionalGRU(rnn_dim * 2, rnn_dim, dropout, batch_first=True) for _ in range(n_rnn_layers - 1)]
        )
        self.classifier = nn.Sequential(
            nn.Linear(rnn_dim * 2, rnn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(rnn_dim, rnn_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(rnn_dim // 2, n_class)
        )
    def forward(self, x):
        x = self.cnn(x); x = self.rescnn_layers(x)
        B, C, freq, T_ = x.size()
        x = x.view(B, C * freq, T_).transpose(1, 2)
        x = self.fully_connected(x); x = self.birnn_layers(x); x = self.classifier(x)
        return F.log_softmax(x, dim=2)

print('✅ Model classes defined.')

✅ Model classes defined.


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
text_transform      = TextTransform()
val_audio_transform = LogMelSpectrogram()

ckpt    = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
hparams = ckpt['hparams']

model = SpeechRecognitionModel(
    hparams['n_cnn_layers'], hparams['n_rnn_layers'], hparams['rnn_dim'],
    hparams['n_class'],      hparams['n_feats'],      hparams['stride'],
    hparams['dropout']
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f"✅ Loaded epoch {ckpt['epoch']}  val_wer={ckpt['val_wer']:.4f}  val_cer={ckpt['val_cer']:.4f}")

/usr/local/lib/python3.12/dist-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (257) may be set too low.
  warnings.warn(


✅ Loaded epoch 26  val_wer=0.6125  val_cer=0.2287


In [7]:
from jiwer import wer, cer
from tqdm import tqdm

def greedy_decode(log_probs):
    pred_idx  = log_probs.argmax(dim=-1).squeeze(0).cpu().tolist()
    blank_idx = text_transform.blank_idx
    collapsed, prev = [], None
    for idx in pred_idx:
        if idx != blank_idx and idx != prev:
            collapsed.append(idx)
        prev = idx
    return text_transform.int_to_text(collapsed)

all_refs, all_greedy, all_logits = [], [], []

for r in tqdm(val_records, desc='Running inference'):
    try:
        waveform, sr = torchaudio.load(r['audio_path'])
        if sr != SAMPLE_RATE:
            waveform = T.Resample(sr, SAMPLE_RATE)(waveform)
        spec = val_audio_transform(waveform)
        spec = (spec - spec.mean()) / (spec.std() + 1e-9)
        spec = spec.unsqueeze(0).to(device)

        with torch.no_grad():
            log_probs = model(spec)

        all_greedy.append(greedy_decode(log_probs) or ' ')
        all_refs.append(r['transcript'].lower().strip())
        all_logits.append(log_probs.squeeze(0).cpu().numpy())
    except Exception as e:
        print(f'⚠️ Skipped {r["audio_path"]}: {e}')
        continue

baseline_preds = all_greedy
print(f"\nGreedy — WER: {wer(all_refs, all_greedy):.4f}  CER: {cer(all_refs, all_greedy):.4f}")

Running inference: 100%|██████████| 2620/2620 [02:40<00:00, 16.35it/s]



Greedy — WER: 0.6091  CER: 0.2267


In [8]:
# Clean any previous partial clone/build before starting
!rm -rf {WORK_DIR}/kenlm
!apt-get update -qq && apt-get install -y -qq build-essential cmake libboost-all-dev libeigen3-dev zlib1g-dev libbz2-dev liblzma-dev
!git clone --recursive https://github.com/kpu/kenlm.git {WORK_DIR}/kenlm
!mkdir -p {WORK_DIR}/kenlm/build
!cd {WORK_DIR}/kenlm/build && cmake .. && make -j 4

# Verify the build actually produced the binaries before continuing
import os
lmplz_path       = f'{WORK_DIR}/kenlm/build/bin/lmplz'
build_binary_path = f'{WORK_DIR}/kenlm/build/bin/build_binary'
assert os.path.exists(lmplz_path), 'lmplz build failed — check cmake/make output above'
assert os.path.exists(build_binary_path), 'build_binary build failed — check cmake/make output above'
print('✅ KenLM built successfully.')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../0-libpython3.10-dev_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-dev:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.15) ...
Preparing to unpack .../1-libpython3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.15) ...
Preparing to unpack .../2-python3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking python3.10 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.15) ...
Preparing to unpack .../3-libpython3.10-stdlib_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-stdlib:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.15) ...
Preparing to unpack .../4-python3.10-minimal_3.10.12-1~22.04.17_amd64.d

In [9]:
!wget -q https://www.openslr.org/resources/11/librispeech-lm-norm.txt.gz -O {WORK_DIR}/lm_text.txt.gz
!zcat {WORK_DIR}/lm_text.txt.gz | head -n {LM_CORPUS_LINES} | tr '[:upper:]' '[:lower:]' > {WORK_DIR}/lm_train.txt
!wc -l {WORK_DIR}/lm_train.txt
!head -n 3 {WORK_DIR}/lm_train.txt   # sanity check — should be lowercase


gzip: stdout: Broken pipe
3000000 /kaggle/working/lm_train.txt

a
a a


In [10]:
!{WORK_DIR}/kenlm/build/bin/lmplz -o 4 < {WORK_DIR}/lm_train.txt > {WORK_DIR}/lm.arpa
!{WORK_DIR}/kenlm/build/bin/build_binary {WORK_DIR}/lm.arpa {WORK_DIR}/lm.binary

lm_binary_path = f'{WORK_DIR}/lm.binary'
assert os.path.exists(lm_binary_path) and os.path.getsize(lm_binary_path) > 0, 'lm.binary was not created — check lmplz/build_binary output above'
print('✅ Language model trained and binary built.')

=== 1/5 Counting and sorting n-grams ===
Reading /kaggle/working/lm_train.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 62825307 types 310899
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:3730788 2:4582770176 3:8592694272 4:13748311040
Statistics:
1 310899 D1=0.653089 D2=1.02397 D3+=1.38647
2 7819811 D1=0.751065 D2=1.05797 D3+=1.33157
3 27493884 D1=0.840075 D2=1.11741 D3+=1.31562
4 45052670 D1=0.896368 D2=1.22549 D3+=1.34189
Memory estimate for binary LM:
type      MB
probing 1589 assuming -p 1.5
probing 1792 assuming -r models -p 1.5
trie     729 without quantization
trie     407 assuming -q 8 -b 8 quantization 
trie     654 assuming -a 22 array pointer compression
trie     332 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabil

In [11]:
SUBSET = 200
best_alpha, best_beta, best_wer = None, None, float('inf')

for alpha in [0.2, 0.5, 1.0]:
    for beta in [0.0, 0.5, 1.5]:
        d = build_ctcdecoder(labels, kenlm_model_path=lm_binary_path, alpha=alpha, beta=beta)
        preds = [d.decode(l) or ' ' for l in all_logits[:SUBSET]]
        w = wer(all_refs[:SUBSET], preds)
        print(f'alpha={alpha}  beta={beta}  ->  WER={w:.4f}')
        if w < best_wer:
            best_wer, best_alpha, best_beta = w, alpha, beta

print(f'\nBest on subset: alpha={best_alpha}  beta={best_beta}  WER={best_wer:.4f}')
print('If this beats the full-set result above, rebuild lm_decoder with these values and re-run Cell 7.')

NameError: name 'build_ctcdecoder' is not defined

In [12]:
from spellchecker import SpellChecker

spell_generic = SpellChecker()

def correct_text(text, checker, max_word_len=15):
    # max_word_len guards against pathologically long/garbled tokens that can make
    # edit-distance correction take minutes on a single malformed word.
    words = text.split()
    corrected = []
    for w in words:
        if len(w) > max_word_len:
            corrected.append(w)
        else:
            corrected.append(checker.correction(w) or w)
    return ' '.join(corrected)

all_dict_generic = [correct_text(p, spell_generic) for p in tqdm(baseline_preds, desc='Generic dict correction')]
print(f"\nGeneric Dictionary — WER: {wer(all_refs, all_dict_generic):.4f}  CER: {cer(all_refs, all_dict_generic):.4f}")

Generic dict correction: 100%|██████████| 2620/2620 [46:54<00:00,  1.07s/it]  



Generic Dictionary — WER: 0.5326  CER: 0.2362


In [13]:
spell_domain = SpellChecker(language=None)  # empty — no generic English bias
spell_domain.word_frequency.load_text_file(f'{WORK_DIR}/lm_train.txt')

all_dict_domain = [correct_text(p, spell_domain) for p in tqdm(baseline_preds, desc='Domain dict correction')]
print(f"\nDomain Dictionary — WER: {wer(all_refs, all_dict_domain):.4f}  CER: {cer(all_refs, all_dict_domain):.4f}")

Domain dict correction: 100%|██████████| 2620/2620 [13:33<00:00,  3.22it/s]



Domain Dictionary — WER: 0.5559  CER: 0.2311


In [14]:
import jellyfish
from collections import Counter

# Build a phonetic code -> most-frequent-word lookup from the domain corpus
word_counts = Counter()
with open(f'{WORK_DIR}/lm_train.txt') as f:
    for line in f:
        word_counts.update(line.split())

phonetic_map = {}
for w, _ in word_counts.most_common():   # most frequent first, so it wins ties
    code = jellyfish.metaphone(w)
    phonetic_map.setdefault(code, w)

def phonetic_correct(text):
    corrected = []
    for w in text.split():
        code = jellyfish.metaphone(w)
        corrected.append(phonetic_map.get(code, w))
    return ' '.join(corrected)

all_phonetic = [phonetic_correct(p) for p in tqdm(baseline_preds, desc='Phonetic correction')]
print(f"\nPhonetic — WER: {wer(all_refs, all_phonetic):.4f}  CER: {cer(all_refs, all_phonetic):.4f}")

Phonetic correction: 100%|██████████| 2620/2620 [00:00<00:00, 81851.66it/s]



Phonetic — WER: 0.5936  CER: 0.3021


In [15]:
all_kenlm_dict = [correct_text(p, spell_domain) for p in tqdm(all_kenlm, desc='KenLM + dict correction')]
print(f"\nKenLM + Domain Dict — WER: {wer(all_refs, all_kenlm_dict):.4f}  CER: {cer(all_refs, all_kenlm_dict):.4f}")

NameError: name 'all_kenlm' is not defined

In [17]:
results = {
    'Greedy (baseline)':        (wer(all_refs, all_greedy),       cer(all_refs, all_greedy)),
    # 'KenLM+Beam':               (wer(all_refs, all_kenlm),        cer(all_refs, all_kenlm)),
    'Generic Dictionary':       (wer(all_refs, all_dict_generic), cer(all_refs, all_dict_generic)),
    'Domain Dictionary':        (wer(all_refs, all_dict_domain),  cer(all_refs, all_dict_domain)),
    'Phonetic Correction':      (wer(all_refs, all_phonetic),     cer(all_refs, all_phonetic)),
    # 'KenLM + Domain Dictionary':(wer(all_refs, all_kenlm_dict),   cer(all_refs, all_kenlm_dict)),
}

print(f"{'Method':<28}{'WER':>8}{'CER':>8}")
print('-' * 44)
for name, (w, c) in sorted(results.items(), key=lambda x: x[1][0]):
    print(f"{name:<28}{w:>8.4f}{c:>8.4f}")

best_method = min(results.items(), key=lambda x: x[1][0])
print(f"\n🏆 Best method: {best_method[0]}  (WER={best_method[1][0]:.4f}, CER={best_method[1][1]:.4f})")

Method                           WER     CER
--------------------------------------------
Generic Dictionary            0.5326  0.2362
Domain Dictionary             0.5559  0.2311
Phonetic Correction           0.5936  0.3021
Greedy (baseline)             0.6091  0.2267

🏆 Best method: Generic Dictionary  (WER=0.5326, CER=0.2362)


In [ ]:
import json

summary = {name: {'wer': w, 'cer': c} for name, (w, c) in results.items()}
with open(f'{WORK_DIR}/postprocessing_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

predictions_dump = {
    'refs': all_refs, 'greedy': all_greedy, 'kenlm': all_kenlm,
    'dict_generic': all_dict_generic, 'dict_domain': all_dict_domain,
    'phonetic': all_phonetic, 'kenlm_dict': all_kenlm_dict,
}
with open(f'{WORK_DIR}/all_predictions.json', 'w') as f:
    json.dump(predictions_dump, f, indent=2)

print('✅ Results and predictions saved.')